In [ ]:
import pandas as pd

df_venture = pd.read_csv(
    "../../../data/기술창업 분석 리포트/벤처기업정보/벤처기업명단_표준산업분류코드매핑_주소포함_prep.xls",
    encoding="utf-8-sig",
)
df_venture['벤처유효시작일'] = pd.to_datetime(df_venture['벤처유효시작일'], format='%Y-%m-%d')
df_venture['벤처유효종료일'] = pd.to_datetime(df_venture['벤처유효종료일'], format='%Y-%m-%d')
print(f"df_venture 로드 완료: {len(df_venture)}행")

## [함수] 유사 벤처 인증 기업 수 

In [ ]:
def count_similar_venture_companies(df_venture: pd.DataFrame, target_codes, code_col: str = '표준산업분류코드') -> pd.DataFrame:
    """
    유사 벤처인증기업 리스트 추출 함수

    사용자 업종과 매핑된 표준산업분류코드를 기준으로,
    전체 벤처기업명단에서 동일/유사 업종 기업만 필터링한다.
    이 함수의 반환값(리스트)이 count_venture_type, 
    compute_venture_type_distribution, compute_venture_density_grid의
    입력으로 재사용된다.

    Parameters
    ----------
    df_venture : pd.DataFrame
        벤처기업명단 전체 데이터
    target_codes : str or list[str]
        국세청 매핑을 거쳐 나온 표준산업분류코드 (여러 개일 수 있음)
    code_col : str
        비교 기준 컬럼명 (기본: 표준산업분류코드)

    Returns
    -------
    pd.DataFrame
        유사 업종에 해당하는 벤처기업 목록 (원본 컬럼 그대로 유지)
    """
    if isinstance(target_codes, str):
        target_codes = [target_codes]

    if code_col not in df_venture.columns:
        raise ValueError(f"'{code_col}' 컬럼이 존재하지 않습니다.")

    similar_companies = df_venture[df_venture[code_col].isin(target_codes)].copy()

    return similar_companies

In [ ]:
# 예시로 J58221(소프트웨어 개발업) 기준 테스트
similar = count_similar_venture_companies(df_venture, target_codes="J58221")
print(f"유사 벤처인증기업 수: {len(similar)}개")
similar.head()

## [함수] 벤처투자형 인증 카운트 (1년)

In [ ]:
import datetime


def count_venture_type(similar_companies: pd.DataFrame, venture_type: str = "벤처투자",
                        period_years: int = 1, reference_date: str = None) -> int:
    """
    유사 벤처기업 중 특정 인증유형 + 최근 N년 이내 신규 인증 카운트

    "이 업종에 최근 투자가 몰리고 있는지"(최근 흐름)를 보여주기 위한 지표.
    누적 전체 건수가 아니라, 벤처유효시작일 기준으로 최근 N년 이내
    신규 인증된 건만 필터링한다.
    (참고: "유사 벤처인증기업 수"는 유형/기간 무관 누적 카운트이고,
    이 함수는 그중 특정 유형 + 최근 기간만 별도로 보는 것 — 서로 다른 지표)

    Parameters
    ----------
    similar_companies : pd.DataFrame
        count_similar_venture_companies()의 반환값
    venture_type : str
        벤처확인유형 값 (혁신성장/벤처투자/연구개발/예비벤처)
        기본값은 "벤처투자" — 실제 투자자(VC 등)로부터 투자를 받아
        인증된 유형으로, "시장에서 실제 투자금이 몰리고 있는지"를
        가장 직접적으로 보여주는 유형이기 때문
    period_years : int
        최근 몇 년 이내로 볼지 (기준: 벤처유효시작일)
    reference_date : str or pd.Timestamp, optional
        기준 날짜 (예: "2026-07-01" 또는 df['컬럼'].max() 같은 Timestamp).
        None이면 코드 실행 시점(오늘) 기준.
        데이터 스냅샷 특성상, 오늘 날짜가 아니라 데이터 안에 실제로
        존재하는 최신 날짜(예: df_venture['벤처유효시작일'].max())를
        넣는 것을 권장 — 파일 발급일과 실제 최신 데이터 시점이
        며칠 이상 어긋날 수 있어 결과가 달라지기 때문

    Returns
    -------
    int
        조건(인증유형 일치 + 최근 N년 이내 시작)에 맞는 기업 수
    """
    # 기준일 설정: 지정 안 하면 오늘 날짜, 지정하면 그 값 사용
    if reference_date:
        ref = pd.Timestamp(reference_date)
    else:
        ref = pd.Timestamp.now()

    # 기준일에서 N년 전 날짜 계산 → 이 날짜 이후 시작된 건만 "최근"으로 인정
    cutoff_date = ref - pd.DateOffset(years=period_years)

    filtered = similar_companies[
        (similar_companies['벤처확인유형'] == venture_type) &
        (similar_companies['벤처유효시작일'] >= cutoff_date)
    ]

    return len(filtered)

In [ ]:
# → 데이터가 실제로 언제까지의 최신 정보를 담고 있는지 확인하기 위함
print(f"벤처유효시작일 최댓값: {df_venture['벤처유효시작일'].max()}")
print(f"벤처유효시작일 최솟값: {df_venture['벤처유효시작일'].min()}")

In [ ]:
# ============================================
# count_venture_type() 최종 테스트 (기준일 자동화 버전)
# 이전엔 reference_date를 "2026-07-01"로 하드코딩했었는데,
# 데이터 안의 실제 최신 벤처유효시작일(2026-07-29)과 이틀 넘게 차이가 나서
# 그만큼 최근 1년 집계 구간이 어긋났었음.
#
# 이제는 하드코딩 대신 데이터에서 최댓값을 직접 가져와 기준일로 사용 →
# 데이터가 나중에 갱신돼도 코드 수정 없이 항상 정확한 "최근 1년" 구간 계산 가능
# ============================================

reference_date = df_venture['벤처유효시작일'].max()
count = count_venture_type(similar, venture_type="벤처투자", period_years=1, reference_date=reference_date)
print(f"최근 1년 내 벤처투자형 인증: {count}건")

## [함수] 유사 기업 인증유형 구성

In [ ]:
def compute_venture_type_distribution(similar_companies: pd.DataFrame) -> pd.DataFrame:
    counts = similar_companies['벤처확인유형'].value_counts().reset_index()
    counts.columns = ['인증유형', '건수']
    counts['비율(%)'] = (counts['건수'] / counts['건수'].sum() * 100).round(1)

    # 반올림 오차 보정: 합계가 100이 되도록 마지막 행에서 차이만큼 조정
    diff = round(100 - counts['비율(%)'].sum(), 1)
    counts.loc[counts.index[-1], '비율(%)'] += diff

    return counts

In [ ]:
distribution = compute_venture_type_distribution(similar)
print(distribution)

In [ ]:
# 비율 합계 (반올림 오차로 99.9~100.1 정도면 정상)
print(f"비율 합계: {distribution['비율(%)'].sum()}%")

# 건수 합계가 원본(similar) 개수와 정확히 일치해야 함
print(f"건수 합계: {distribution['건수'].sum()}건 (similar 원본: {len(similar)}건)")

## [함수] 동종산업 밀집도

In [ ]:
def compute_venture_density_grid(similar_companies: pd.DataFrame, grid_cols: int = 6, grid_rows: int = 3) -> dict:
    """
    동종산업 밀집도 격자화 함수 (시군구 단위)

    벤처기업명단에는 위경도가 없어 물리적 반경 계산이 불가능하므로,
    실제 지리적 좌표가 아닌 "시군구별 유사기업 개수"를 격자 형태로
    배치하는 추상적 시각화용 함수.
    시군구를 유사기업 수가 많은 순으로 정렬한 뒤 격자 칸에 순서대로 배치한다
    (실제 지리적 위치와는 무관 - 화면 문구 "개별 성공 확률 아님"과 일치하는 개념).

    Parameters
    ----------
    similar_companies : pd.DataFrame
        count_similar_venture_companies()의 반환값 (시군구 컬럼 포함된 데이터 기준)
    grid_cols : int
        격자 가로 칸 수 (기본 6, 화면 시안 기준)
    grid_rows : int
        격자 세로 칸 수 (기본 3, 화면 시안 기준)

    Returns
    -------
    dict
        {
            "grid_cols": int,
            "grid_rows": int,
            "cells": [{"x":.., "y":.., "count":.., "sigungu":..}, ...],  # 상위 grid_cols*grid_rows개만 포함
            "total_sigungu_count": int,
            "shown_sigungu_count": int,
            "total_company_count": int,
            "shown_company_count": int
        }
    """
    if '시군구' not in similar_companies.columns:
        raise ValueError("'시군구' 컬럼이 존재하지 않습니다.")

    sigungu_counts = (
        similar_companies['시군구']
        .value_counts()
        .reset_index()
    )
    sigungu_counts.columns = ['시군구', '건수']

    max_cells = grid_cols * grid_rows
    cells = []
    for idx, row in sigungu_counts.iterrows():
        if idx >= max_cells:
            break
        x = idx % grid_cols
        y = idx // grid_cols
        cells.append({
            "x": x,
            "y": y,
            "count": int(row['건수']),
            "sigungu": row['시군구']
        })

    return {
        "grid_cols": grid_cols,
        "grid_rows": grid_rows,
        "cells": cells,
        "total_sigungu_count": len(sigungu_counts),
        "shown_sigungu_count": len(cells),
        "total_company_count": int(sigungu_counts['건수'].sum()),
        "shown_company_count": sum(c['count'] for c in cells)
    }

In [ ]:
density_result_venture = compute_venture_density_grid(similar, grid_cols=6, grid_rows=3)

print(f"전체 시군구 수: {density_result_venture['total_sigungu_count']}개")
print(f"화면에 표시된 시군구 수: {density_result_venture['shown_sigungu_count']}개")
print(f"전체 기업 수: {density_result_venture['total_company_count']}건")
print(f"화면에 표시된 기업 수: {density_result_venture['shown_company_count']}건")

## [데이터 검증] 벤처기업명단
** 참고용

In [ ]:
# import pandas as pd

# df_venture = pd.read_csv("../../../data/기술창업 분석 리포트/벤처기업정보/벤처기업명단_표준산업분류코드매핑_주소포함_prep.xls", encoding="utf-8-sig")

# # 1. 기본 정보
# print(f"전체 행 수: {len(df_venture)}")
# print(df_venture.columns.tolist())

# # 2. 결측치 확인 (핵심 컬럼 위주)
# print("\n=== 결측치 개수 ===")
# print(df_venture[['업체명', '벤처확인유형', '표준산업분류코드', '벤처유효시작일', '벤처유효종료일']].isnull().sum())

# # 3. 벤처확인유형 종류 확인 (인증유형 구성 비율 계산에 쓸 값들)
# print("\n=== 벤처확인유형 종류 ===")
# print(df_venture['벤처확인유형'].value_counts())

# # 4. 표준산업분류코드 형식 확인 (앞자리 알파벳 + 숫자 형식이 상권데이터와 동일한지)
# print("\n=== 표준산업분류코드 샘플 ===")
# print(df_venture['표준산업분류코드'].unique()[:10])

# # 5. 날짜 컬럼 타입 확인 (문자열인지 날짜형인지 - "1년 이내" 필터링에 필요)
# print("\n=== 날짜 컬럼 타입 ===")
# print(df_venture[['벤처유효시작일', '벤처유효종료일']].dtypes)
# print(df_venture[['벤처유효시작일', '벤처유효종료일']].head())

In [ ]:
# # 날짜 컬럼을 datetime으로 변환
# df_venture['벤처유효시작일'] = pd.to_datetime(df_venture['벤처유효시작일'], format='%Y-%m-%d')
# df_venture['벤처유효종료일'] = pd.to_datetime(df_venture['벤처유효종료일'], format='%Y-%m-%d')

# # 변환 확인
# print(df_venture[['벤처유효시작일', '벤처유효종료일']].dtypes)
# print(df_venture[['벤처유효시작일', '벤처유효종료일']].head())